# Preparación del dataset supervisado de actividad FIRMS

Combina el target multiclase con meteorología, contexto y antecedentes FIRMS estrictamente anteriores. Las filas objetivo son 2018–2025; 2017 se usa sólo como historia previa.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
HIST=Path("data/processed/firms_open_meteo_weekly/open_meteo_firms_weekly_history.parquet")
TARGET=Path("data/processed/modelado/dataset_niveles_actividad_firms.parquet")
SALIDA=Path("data/processed/modelado/dataset_modelado_nivel_actividad_firms.parquet")
CLAVES=["departamento","fecha_inicio_semana"]
hist=pd.read_parquet(HIST).copy(); target=pd.read_parquet(TARGET).copy()
for x in (hist,target):x["fecha_inicio_semana"]=pd.to_datetime(x.fecha_inicio_semana)
print("Historia",hist.shape,hist.fecha_inicio_semana.min().date(),hist.fecha_inicio_semana.max().date());print("Target",target.shape)

Historia (8911, 35) 2017-01-02 2025-12-22
Target (7923, 32)


## 1. Lags FIRMS sin fuga

Se agrupa por departamento y se desplaza antes de cualquier rolling. Ninguna variable incorpora el conteo de la semana objetivo.

In [2]:
hist=hist.sort_values(CLAVES).reset_index(drop=True)
assert not hist.duplicated(CLAVES).any() and hist.groupby("departamento").fecha_inicio_semana.diff().dropna().dt.days.eq(7).all()
g=hist.groupby("departamento",sort=False).cantidad_detecciones_hist
for lag in range(1,5):hist[f"cantidad_detecciones_lag{lag}"]=g.shift(lag)
lag1=hist.cantidad_detecciones_lag1
hist["promedio_detecciones_ultimas_4_semanas"]=lag1.groupby(hist.departamento).rolling(4,min_periods=4).mean().reset_index(level=0,drop=True)
hist["max_detecciones_ultimas_4_semanas"]=lag1.groupby(hist.departamento).rolling(4,min_periods=4).max().reset_index(level=0,drop=True)
hist["semanas_con_actividad_ultimas_4"]=lag1.gt(0).astype(int).groupby(hist.departamento).rolling(4,min_periods=4).sum().reset_index(level=0,drop=True)

## 2. Comprobación manual

In [3]:
LAGS=[f"cantidad_detecciones_lag{i}" for i in range(1,5)]; ROLL=["promedio_detecciones_ultimas_4_semanas","max_detecciones_ultimas_4_semanas","semanas_con_actividad_ultimas_4"]
ej=hist.loc[hist.departamento.eq(hist.departamento.iloc[0])].reset_index(drop=True);i=10;ant=ej.loc[i-4:i-1,"cantidad_detecciones_hist"].iloc[::-1].to_numpy()
assert np.array_equal(ej.loc[i,LAGS].to_numpy(float),ant.astype(float));assert ej.loc[i,ROLL[0]]==ant.mean();assert ej.loc[i,ROLL[1]]==ant.max();assert ej.loc[i,ROLL[2]]==(ant>0).sum()
pd.DataFrame([ant],columns=LAGS,index=[f"Semana objetivo {ej.loc[i,'fecha_inicio_semana'].date()}"])

,cantidad_detecciones_lag1,cantidad_detecciones_lag2,cantidad_detecciones_lag3,cantidad_detecciones_lag4
Semana objetivo 2017-03-13,0,0,0,0


## 3. Integración y predictores

El conteo actual queda sólo para auditar el target. La meteorología contemporánea supone disponibilidad durante la semana o pronósticos equivalentes en una aplicación operativa.

In [4]:
METEO=["temperature_2m_max_mean_weekly","relative_humidity_2m_min_mean_weekly","wind_speed_10m_max_mean_weekly","precipitation_sum_weekly"]
cols=CLAVES+["anio","numero_semana"]+METEO+LAGS+ROLL
base=hist[cols].merge(target[CLAVES+["cantidad_detecciones","nivel_actividad_firms","nivel_actividad_firms_codigo"]],on=CLAVES,validate="one_to_one")
base["mes"]=base.fecha_inicio_semana.dt.month.astype("int8")
assert len(base)==7923 and base.anio.between(2018,2025).all() and not base.duplicated(CLAVES).any()
assert not base[LAGS+ROLL].isna().any().any()

In [5]:
CATEGORICAS=["departamento","mes"];NUMERICAS=METEO+LAGS+ROLL;PREDICTORES=CATEGORICAS+NUMERICAS
PROHIBIDAS={"cantidad_detecciones","cantidad_detecciones_hist","nivel_actividad_firms","nivel_actividad_firms_codigo"}
assert PROHIBIDAS.isdisjoint(PREDICTORES) and not base[PREDICTORES].isna().any().any()
pd.DataFrame({"predictor":PREDICTORES,"tipo":["categórica" if x in CATEGORICAS else "numérica" for x in PREDICTORES]})

,predictor,tipo
0,departamento,categórica
1,mes,categórica
2,temperature_2m_max_mean_weekly,numérica
3,relative_humidity_2m_min_mean_weekly,numérica
4,wind_speed_10m_max_mean_weekly,numérica
5,precipitation_sum_weekly,numérica
6,cantidad_detecciones_lag1,numérica
7,cantidad_detecciones_lag2,numérica
8,cantidad_detecciones_lag3,numérica
9,cantidad_detecciones_lag4,numérica


In [6]:
salida_cols=list(dict.fromkeys(CLAVES+["anio","numero_semana","mes"]+PREDICTORES+["cantidad_detecciones","nivel_actividad_firms","nivel_actividad_firms_codigo"]))
dataset=base[salida_cols].sort_values(CLAVES).reset_index(drop=True);SALIDA.parent.mkdir(parents=True,exist_ok=True);dataset.to_parquet(SALIDA,index=False)
print("Guardado",SALIDA,dataset.shape);print(dataset.nivel_actividad_firms.value_counts())

Guardado data/processed/modelado/dataset_modelado_nivel_actividad_firms.parquet (7923, 19)
nivel_actividad_firms
Sin actividad    4967
Moderado         1266
Bajo             1237
Alto              453
Name: count, dtype: int64


## Conclusión

Las 7.923 filas conservan siete antecedentes FIRMS: lags 1–4, media y máximo previos de cuatro semanas y número de semanas activas previas. El histórico 2017 evita nulos al inicio de 2018. Ningún predictor utiliza FIRMS de la semana objetivo.